In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid')


In [ ]:
data_path = '../EV_Charging_Demand_Final.csv'
df = pd.read_csv(data_path)

print(df.head())
print('\nShape:', df.shape)
print('\nData types:')
print(df.dtypes)


In [ ]:
df = df.copy()
for col in ['charging_sessions', 'energy_kwh', 'peak_demand_kw', 'temperature_c']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype('string').str.strip()

print(df.isnull().sum())

df = df.dropna(subset=['date', 'charging_sessions', 'energy_kwh', 'peak_demand_kw'])


In [ ]:
print(df.describe())

plt.figure(figsize=(10, 4))
sns.histplot(df['charging_sessions'], bins=12, kde=True)
plt.title('Distribution of Charging Sessions')
plt.xlabel('Charging Sessions')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(10, 4))
sns.scatterplot(data=df, x='temperature_c', y='peak_demand_kw', alpha=0.7)
plt.title('Peak Demand vs Temperature')
plt.xlabel('Temperature (°C)')
plt.ylabel('Peak Demand (kW)')
plt.show()


In [ ]:
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()
df['month'] = df['date'].dt.month

df['demand_per_session'] = df['energy_kwh'] / df['charging_sessions']

print(df[['hour', 'day_of_week', 'month', 'demand_per_session']].head())


In [ ]:
df_ts = df.set_index('date').resample('D')['charging_sessions'].sum().to_frame()
df_ts.plot(figsize=(12, 4), title='Daily Charging Sessions Trend')
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x='day_of_week', y='charging_sessions')
plt.title('Charging Sessions by Day of Week')
plt.xticks(rotation=45)
plt.show()


In [ ]:
features = ['hour', 'month', 'temperature_c']
target = 'charging_sessions'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')

plt.figure(figsize=(8, 4))
plt.scatter(y_test, predictions, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Sessions')
plt.ylabel('Predicted Sessions')
plt.title('Actual vs Predicted Charging Sessions')
plt.show()


In [ ]:
df_processed = df.copy()
df_processed.to_csv('../processed_EV_charging_demand.csv', index=False)
print('Saved processed data to ../processed_EV_charging_demand.csv')
